# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q emoji wordcloud textblob
    print('Packages installed.')


# WhatsApp Chat Analysis In-Depth

This notebook provides a comprehensive toolkit for analyzing exported WhatsApp chat data. Unlike basic tutorials, this notebook covers:

1.  **Robust Data Generation**: Creating a realistic synthetic dataset for testing.
2.  **Advanced Parsing**: Handling multiple device formats (iOS/Android) and multi-line messages.
3.  **Temporal Analysis**: Activity heatmaps (Day x Hour) and response time distributions.
4.  **Content Analysis**: Sentiment tracking, Emoji frequency, and Word Clouds.
5.  **Interaction Network**: visualizing who replies to whom.

## Libraries Used
- `pandas`: Data manipulation.
- `seaborn` / `matplotlib`: Visualization.
- `emoji`: Extracting and counting emojis.
- `textblob`: Sentiment analysis.
- `wordcloud`: Visualizing frequent terms.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import random
from datetime import datetime, timedelta
import emoji
from collections import Counter
from textblob import TextBlob
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Realistic Synthetic Data Generation

To ensure this notebook is fully functional without requiring you to upload private data immediately, we generate a comprehensive dummy chat log. This simulator creates meaningful patterns (e.g., sleep gaps, bursty conversations) rather than just random timestamps.

In [ ]:
def generate_dummy_chat(num_messages=500):
    users = ['Alice', 'Bob', 'Charlie', 'Diana']
    
    # Templates for meaningful text
    templates = [
        "Hey guys, how is it going?", "Did you see the news? 😱",
        "I am running late!", "Meeting is at 5pm.",
        "Hahaha that is so funny 😂", "Can you send me the file?",
        "Thanks!", "Good morning! ☀️", "Good night everyone 💤",
        "What are we having for lunch?", "I'm busy right now.",
        "Let's catch up later.", "This project is going well.",
        "I disagree with that point.", "Great job! 👍"
    ]
    
    messages = []
    current_time = datetime(2023, 1, 1, 9, 0, 0)
    
    for _ in range(num_messages):
        # Determine time gap: mostly short (replies), sometimes long (new convo)
        if random.random() < 0.8:
            # Reply: 10 seconds to 5 minutes
            gap = timedelta(seconds=random.randint(10, 300))
        else:
            # New topic or break: 1 hour to 12 hours
            gap = timedelta(hours=random.randint(1, 12))
            
        current_time += gap
        
        # Skip sleeping hours (roughly 1 AM to 7 AM) unless random insomniac event
        if 1 <= current_time.hour < 7 and random.random() > 0.1:
            current_time += timedelta(hours=6)
            
        sender = random.choice(users)
        message = random.choice(templates)
        
        # Randomly format as iOS or Android
        if random.choice(['ios', 'android']) == 'ios':
            # [12/04/2023, 14:30:15] User: Message
            timestamp = current_time.strftime("%d/%m/%Y, %H:%M:%S")
            line = f"[{timestamp}] {sender}: {message}"
        else:
             # 12/04/2023, 14:30 - User: Message
            timestamp = current_time.strftime("%d/%m/%Y, %H:%M")
            line = f"{timestamp} - {sender}: {message}"
            
        messages.append(line)
        
    return messages

# Generate and save
chat_data = generate_dummy_chat(800)
with open('dummy_chat.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(chat_data))
    
print(f"Generated {len(chat_data)} dummy messages.")
print("Sample lines:")
print(chat_data[:3])

## 2. Robust Parsing Logic

WhatsApp exports vary by device (iOS uses brackets `[]`, Android uses dashes `-`). Our parser uses regular expressions to detect the format automatically.

In [ ]:
def parse_whatsapp_chat(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        data = f.readlines()
        
    messages = []
    # Regex patterns
    # Android: 12/04/2023, 14:30 - User: Message
    android_regex = r'^(\d{1,2}/\d{1,2}/\d{2,4}), (\d{1,2}:\d{2}) - (.*?): (.*)$'
    # iOS: [12/04/2023, 14:30:15] User: Message
    ios_regex = r'^\[(\d{1,2}/\d{1,2}/\d{2,4}), (\d{1,2}:\d{2}:\d{2})\] (.*?): (.*)$'
    
    current_date = None
    
    for line in data:
        line = line.strip()
        if not line:
            continue
            
        # Try Android
        match = re.match(android_regex, line)
        if match:
            date, time, sender, message = match.groups()
            messages.append([date, time, sender, message])
            continue
            
        # Try iOS
        match = re.match(ios_regex, line)
        if match:
            date, time, sender, message = match.groups()
            messages.append([date, time, sender, message])
            continue
            
        # Handle multi-line messages (appending to previous message)
        if messages:
            messages[-1][-1] += " " + line

    df = pd.DataFrame(messages, columns=['Date', 'Time', 'Sender', 'Message'])
    
    # Convert to DateTime
    # Handling potentially mixed formats if multiple files were merged
    df['timestamp'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True, errors='coerce')
    
    # Drop messages where timestamp parsing failed (system messages often fail these regexes)
    df.dropna(subset=['timestamp'], inplace=True)
    
    return df

df = parse_whatsapp_chat('dummy_chat.txt')
print(f"Parsed {len(df)} messages successfully.")
print(df.head())

## 3. Basic Activity Stats
Who talks the most? What is the average message length?

In [ ]:
df['Message_Length'] = df['Message'].apply(len)

user_stats = df.groupby('Sender').agg(
    Total_Messages=('Message', 'count'),
    Avg_Length=('Message_Length', 'mean')
).sort_values('Total_Messages', ascending=False)

print(user_stats)

# Visualize
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=user_stats.index, y=user_stats['Total_Messages'], ax=ax[0], palette='viridis')
ax[0].set_title('Total Messages by User')

sns.boxplot(data=df, x='Sender', y='Message_Length', ax=ax[1], palette='viridis')
ax[1].set_title('Message Length Distribution')
ax[1].set_yscale('log') # Log scale helps view outliers
plt.show()

## 4. Response Time Analysis
We measure how quickly users respond to the *previous* message. We define a "conversation break" as any gap larger than 1 hour.

In [ ]:
# Calculate time difference
df['prev_timestamp'] = df['timestamp'].shift(1)
df['response_time_min'] = (df['timestamp'] - df['prev_timestamp']).dt.total_seconds() / 60

# Filter: Ignore first message and long gaps (> 60 mins) which imply a new session
responses = df[df['response_time_min'] < 60]

avg_response = responses.groupby('Sender')['response_time_min'].mean().sort_values()

plt.figure(figsize=(8, 4))
avg_response.plot(kind='barh', color='skyblue')
plt.title('Average Response Time (Minutes)')
plt.xlabel('Minutes')
plt.show()

## 5. Activity Heatmap
When is the group most active?

In [ ]:
df['DayOfWeek'] = df['timestamp'].dt.day_name()
df['Hour'] = df['timestamp'].dt.hour

heatmap_data = df.pivot_table(index='DayOfWeek', columns='Hour', values='Message', aggfunc='count', fill_value=0)

days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = heatmap_data.reindex(days_order)

plt.figure(figsize=(12, 6))
sns.heatmap(heatmap_data, cmap='YlGnBu', annot=False)
plt.title('Activity Heatmap')
plt.show()

## 6. Emoji Analysis
Extracting emojis to see which are most popular.

In [ ]:
def extract_emojis(text):
    return [c for c in text if c in emoji.EMOJI_DATA]

all_emojis = []
for msg in df['Message']:
    all_emojis.extend(extract_emojis(msg))

emoji_counts = Counter(all_emojis)
top_emojis = emoji_counts.most_common(10)

print("Top 10 Emojis:")
for e, count in top_emojis:
    print(f"{e}: {count}")

if top_emojis:
    ems, counts = zip(*top_emojis)
    plt.figure(figsize=(10, 5))
    plt.bar(range(len(ems)), counts, tick_label=ems)
    plt.title('Top 10 Emojis')
    plt.show()

## 7. Sentiment Timeline
Are conversations getting happier or more serious over time? We use a rolling average to smooth out the noise.

In [ ]:
df['sentiment'] = df['Message'].apply(lambda x: TextBlob(x).sentiment.polarity)

# Resample by day to see trends
daily_sentiment = df.set_index('timestamp').resample('D')['sentiment'].mean()

plt.figure(figsize=(12, 5))
daily_sentiment.plot(color='purple', lw=2)
plt.axhline(0, color='gray', linestyle='--')
plt.title('Daily Average Sentiment')
plt.ylabel('Polarity (-1 to +1)')
plt.show()